In [1]:
!rm -rf .five_minute_cache

# Fleche in Five Minutes

*A persistent cache for expensive Python functions.*

Decorate a function and `fleche` stores every result under a SHA256 key built from the
function's identity and the **content** of its arguments. Results survive restarts, can
live in files, HDF5, SQL, or on another machine, and everything you ever computed stays
queryable like a small database.

If your day involves functions that take minutes to days (structure relaxations,
phonons, MD, training runs) and you re-run them more often than you'd like, this is
for you.

```
pip install fleche          # or: conda install -c conda-forge fleche
pip install ase fleche-ase executorlib      # for the ASE and executor sections
```

## 1. Decorate and forget

Point the active cache at a directory (in real projects you'd do this once in a
`fleche.toml` next to your code; see the end of this notebook), then just decorate:

In [2]:
import time
from fleche import fleche, cache, tags, wrap_executor
from fleche.caches import Cache

cache(Cache.from_config({"template": "cloudpickle", "root": "./.five_minute_cache"}));

In [3]:
@fleche
def expensive(x):
    print(f"crunching {x} ...")
    time.sleep(2)                     # pretend this is a real calculation
    return x ** 2


start = time.time()
print(expensive(4), f"  <- first call: {time.time() - start:.2f} s")

start = time.time()
print(expensive(4), f"  <- second call: {time.time() - start:.4f} s")

crunching 4 ...
16   <- first call: 2.00 s
16   <- second call: 0.0009 s


In [4]:
ls .five_minute_cache/calls/*

.five_minute_cache/calls/6bd4a06377ba5fe070ae8efc9f652637332ad07fa9db0ab9b88eb441ee301670


In [5]:
ls .five_minute_cache/values/*

.five_minute_cache/values/83ada2198553b88cb3d0882f7fca8c4e9531049b978df3e9e3b5d6301c6c0bfa
.five_minute_cache/values/ab7147c672380ae7a61133670895102a6a22971e1aaaefa270a166334d06a49a


The second call never enters the function body. Unlike `lru_cache`, the result is on
disk, not in process memory. Restart the kernel (skip the cleanup cell at the top) and
it is still instant. A brand-new cache object pointed at the same directory already
knows the answer:

In [6]:
fresh = Cache.from_config({"template": "cloudpickle", "root": "./.five_minute_cache"})
with cache(fresh):
    print("cached?", expensive.fleche.contains(4))

cached? True


The cache is just a directory: relocatable or sharable.

## 2. Keys are content, not object identity

Arguments are digested by **value**, not `id()`, not pickle bytes. NumPy arrays,
pandas frames, dataclasses, and nested containers work out of the box; equal content
means equal key, no matter who constructed the object:

In [7]:
import numpy as np


@fleche
def norm(arr):
    print("computing ...")
    return float(np.linalg.norm(arr))


a = np.linspace(0, 1, 1_000_000)
print(norm(a))
print(norm(a.copy()))                 # different object, same content -> cache hit

computing ...
577.3504135273193
577.3504135273193



The digest machinery is pluggable for third-party types. 
For ASE, there is `fleche-ase`.

In [ ]:
try:
    from ase.build import bulk
    from ase.calculators.emt import EMT
except ImportError:
    print("ase (and fleche-ase) not installed, skipping this section")
else:
    @fleche
    def energy(atoms):
        print("running EMT ...")
        atoms = atoms.copy()
        atoms.calc = EMT()
        return atoms.get_potential_energy()


    print(energy(bulk("Cu", cubic=True)))
    print(energy(bulk("Cu", cubic=True)))   # freshly built Atoms -> cache hit

Methods work too, without special handling: `self` is just another argument, digested by
content like any other. Two instances carrying the same state share cache entries, and
mutating the state you care about invalidates them:

In [9]:
from dataclasses import dataclass


@dataclass
class Simulation:
    temperature: float
    steps: int = 10

    @fleche
    def run(self, label):
        print(f"  ...actually running {label} at {self.temperature} K")
        return self.temperature * self.steps


print(Simulation(300.0).run("heat"))
print(Simulation(300.0).run("heat"))    # a different object, same state -> hit
print(Simulation(400.0).run("heat"))    # different state -> miss

  ...actually running heat at 300.0 K
3000.0
3000.0
  ...actually running heat at 400.0 K
4000.0


Like every function, this requires 'pure' functions, ie. no side-effects. If the method changes instance attributes, fleche **won't** replay those.

### What goes into the key

Name, module, and arguments, but *not* the function body, so editing a function keeps
serving the old results. Opt in with `hash_code=True` and every source edit invalidates
its entries:

In [10]:
@fleche(hash_code=True)
def free_energy(T):
    return -1.5 * T                       # first version of the model


print(free_energy(300), "  cached?", free_energy.fleche.contains(300))


@fleche(hash_code=True)
def free_energy(T):                       # same name, corrected model
    return -1.5 * T - 0.002 * T ** 2


print("after editing the body, cached?", free_energy.fleche.contains(300))
print(free_energy(300))

-450.0   cached? True
after editing the body, cached? False
-630.0


This hashes *this* function's source, you can explicitly invalidate entries or force to keep the cache key equal by setting 
`@fleche(version=...)`.
`ignore=` and `require=` control which arguments take part in the key at all.
Results for multiple different versions and code hashes can be part of the the same cache.

## 3. Your cache is a database

Every call is recorded with its arguments, runtime, and metadata, kept *separately* from
the (possibly heavy) result values, so you can browse what you computed without
deserializing any of it. Tag calls, then query into a pandas DataFrame:

In [11]:
@fleche
def relax(a, k):
    time.sleep(0.1)                   # pretend
    return {"energy": -a * k, "volume": a ** 3}


with tags(project="five-minute-demo"):
    for a_lat in (3.5, 3.6, 3.7):
        relax(a_lat, k=4)

relax.fleche.query().table(arguments=["a", "k"], results=True)

,name,module,result,timestart,timestop,walltime,project,a,k
ad2b,relax,__main__,"{'energy': -14.0, 'volume': 42.875}",2026-08-10 08:49:05.949266672-04:00,2026-08-10 08:49:06.049993515-04:00,0.100727,five-minute-demo,3.5,4
ba58,relax,__main__,"{'energy': -14.8, 'volume': 50.653000000000006}",2026-08-10 08:49:06.152344465-04:00,2026-08-10 08:49:06.252923727-04:00,0.100579,five-minute-demo,3.7,4
0275,relax,__main__,"{'energy': -14.4, 'volume': 46.656000000000006}",2026-08-10 08:49:06.050945997-04:00,2026-08-10 08:49:06.151510239-04:00,0.100564,five-minute-demo,3.6,4


Cache is queryable, to filter and sort results or to check what's already been computed.

In [12]:
target = 3.68
close = (
    relax.fleche.query()
    .filter(lambda c: abs(c.arguments["a"] - target) < 0.1)
    .sorted(key=lambda c: abs(c.arguments["a"] - target))
)
print(f"nothing cached for a={target}, but {close.count()} nearby run(s):")
close.table(arguments=["a", "k"], results=True)

nothing cached for a=3.68, but 2 nearby run(s):


,name,module,result,timestart,timestop,walltime,project,a,k
ba58,relax,__main__,"{'energy': -14.8, 'volume': 50.653000000000006}",2026-08-10 08:49:06.152344465-04:00,2026-08-10 08:49:06.252923727-04:00,0.100579,five-minute-demo,3.7,4
0275,relax,__main__,"{'energy': -14.4, 'volume': 46.656000000000006}",2026-08-10 08:49:06.050945997-04:00,2026-08-10 08:49:06.151510239-04:00,0.100564,five-minute-demo,3.6,4


Queries chain (`filter`, `sorted`, `unique`, `groupby`, ...) and terminal methods can
`transfer()` matching entries to another cache or `evict()` them. And the view is not
per-function or per-project. `cache().table()` spans everything that ever wrote to this
cache:

In [13]:
cache().table()

,name,module,timestart,timestop,walltime,project
31e0,energy,__main__,2026-08-10 08:49:05.911141634-04:00,2026-08-10 08:49:05.913827658-04:00,0.002686,NaN
fcbd,norm,__main__,2026-08-10 08:49:05.463237286-04:00,2026-08-10 08:49:05.472317219-04:00,0.009080,NaN
ad2b,relax,__main__,2026-08-10 08:49:05.949266672-04:00,2026-08-10 08:49:06.049993515-04:00,0.100727,five-minute-demo
8e7d,free_energy,__main__,2026-08-10 08:49:05.936557293-04:00,2026-08-10 08:49:05.936736107-04:00,0.000179,NaN
ba58,relax,__main__,2026-08-10 08:49:06.152344465-04:00,2026-08-10 08:49:06.252923727-04:00,0.100579,five-minute-demo
b439,Simulation.run,__main__,2026-08-10 08:49:05.928089380-04:00,2026-08-10 08:49:05.930385351-04:00,0.002296,NaN
0275,relax,__main__,2026-08-10 08:49:06.050945997-04:00,2026-08-10 08:49:06.151510239-04:00,0.100564,five-minute-demo
6bd4,expensive,__main__,2026-08-10 08:49:03.205544472-04:00,2026-08-10 08:49:05.206341982-04:00,2.000798,NaN
23f1,free_energy,__main__,2026-08-10 08:49:05.935136557-04:00,2026-08-10 08:49:05.935358524-04:00,0.000222,NaN
7104,Simulation.run,__main__,2026-08-10 08:49:05.918736458-04:00,2026-08-10 08:49:05.925585270-04:00,0.006849,NaN


## 4. Storing everything without filling the disk

Values are content-addressed, so equal results are stored once no matter how many calls
produced them. That is not a corner case. Think of relaxations converging to the same
minimum from different starting points:

In [14]:
from pathlib import Path


def cache_size(root="./.five_minute_cache"):
    return sum(f.stat().st_size for f in Path(root).rglob("*") if f.is_file())


@fleche
def relax_structure(scale):
    # a real relaxation would iterate here; all these starting points fall into
    # the same minimum, so every call returns the same positions
    return np.full((2000, 3), 3.61)


before = cache_size()
scales = (0.9, 0.95, 1.0, 1.05, 1.1)
raw = sum(relax_structure(s).nbytes for s in scales)
print(f"{len(scales)} results, {raw / 1024:.0f} KiB of arrays"
      f"  ->  cache grew by {(cache_size() - before) / 1024:.0f} KiB")

5 results, 234 KiB of arrays  ->  cache grew by 49 KiB


To clear values from a cache, evicting call records leaves the values behind, and `gc()` then sweeps
anything without a reference.  Needs to be done *manually*!

In [15]:
relax_structure.fleche.query().evict()               # drop those call records
print("value entries reclaimed by gc():", len(cache().gc()))
print("earlier results untouched:", relax.fleche.contains(3.5, k=4))

value entries reclaimed by gc(): 6
earlier results untouched: True


## 5. Plays well with executors, including executorlib

`wrap_executor` patches any `concurrent.futures`-style executor. Fleche-decorated
functions carry the active cache into worker processes automatically, and cache hits
come back as already-completed futures **without ever being submitted**:

In [16]:
from executorlib import SingleNodeExecutor as Executor


@fleche
def md_step(x):
    time.sleep(1)                     # pretend
    return x ** 3


for attempt in ("cold", "warm"):
    start = time.time()
    with Executor(max_workers=4) as ex:
        wrap_executor(ex)
        results = [f.result() for f in [ex.submit(md_step, x) for x in range(4)]]
    print(f"{attempt}: {results} in {time.time() - start:.2f} s")

cold: [0, 1, 8, 27] in 1.79 s
warm: [0, 1, 8, 27] in 0.02 s


On the warm pass every future is done before `submit` returns, so the executor never
sees the work. Executor-specific kwargs like executorlib's `resource_dict` are forwarded
transparently.

## 6. Stack caches: a read-only, prepopulated base

Say you already have a cache full of results, on a shared filesystem. 
`CacheStack`/`Cache.push`. puts your own writable cache *in front* of it: loads
fall through to the base and hits are back-filled into the front, while saves only ever
touch the front. Wrap the base in `ReadOnlyCache` and nothing you do can modify it,
since writes and evictions raise `Rejected`:

In [17]:
from fleche.caches import BaseCache

@fleche
def phonon_dos(structure):
    print(f"computing DOS for {structure} ...")
    time.sleep(1)                     # pretend
    return f"dos({structure})"


# an external prepopulated cache (in memory here; files or SSH in real life)
shared = BaseCache.from_config({"template": "memory"})
with cache(shared):
    phonon_dos("fcc-Al")

mine = BaseCache.from_config({"template": "memory"})
stack = shared.readonly().push(mine)

with cache(stack):
    start = time.time()
    print(phonon_dos("fcc-Al"), f"  <- from the shared base: {time.time() - start:.3f} s")
    print(phonon_dos("bcc-Fe"), "  <- not in the base: computed here")

computing DOS for fcc-Al ...
dos(fcc-Al)   <- from the shared base: 0.000 s
computing DOS for bcc-Fe ...
dos(bcc-Fe)   <- not in the base: computed here


In [18]:
with cache(mine):
    print("back-filled into my cache:", phonon_dos.fleche.contains("fcc-Al"))
    print("new result in my cache:   ", phonon_dos.fleche.contains("bcc-Fe"))
with cache(shared):
    print("shared base untouched:    ", not phonon_dos.fleche.contains("bcc-Fe"))

back-filled into my cache: True
new result in my cache:    True
shared base untouched:     True


Cache configuration can get cumbersome, so there's a config file for permanent setups.

The same stack straight from `fleche.toml`, an array of tables whose first entry is the
front where saves land:

```toml
[[stacked]]
template = "memory"

[[stacked]]
template = "cloudpickle"
root = "/groupshare/fleche-cache"
read_only = true
```

## 7. Things worth a second look

- **Configuration by file**: drop a `fleche.toml` next to your project (or in `$HOME`);
  decorated code never changes:

  ```toml
  [default]
  cache = "persistent"

  [persistent]
  template = "cloudpickle"
  root = "~/.cache/fleche"
  ```

- **HDF5 storage**: `template = "bagofholding_hdf"` stores values via pyiron's
  [bagofholding](https://github.com/pyiron/bagofholding), for storage meant to outlive
  the environment that wrote it.
- **SQL call index**: keep call records in SQLite/Postgres for server-side filtering
  over large caches, while values stay in files or HDF5.
- **Caches on other machines**: `SshCache` forwards a whole cache over SSH to a
  `python -m fleche remote --serve` process on, say, your cluster's login node. Stack it
  behind a local cache exactly as above, or fan reads over several caches at
  once with a read-only `CachePool`.
- **Take results elsewhere**: `query().transfer(other_cache)` moves selected results
  between caches (cluster to laptop); see `TransferWorkflow.ipynb`.
- **Ship a warm cache**: because a cache is just a directory, you can hand one to CI or
  anyone running a notebooks that would otherwise need a supercomputer replay instantly.
- **Inside pyiron_workflow**: `@fleche` and `@pyiron_workflow.atomic` do not interfere,
  and passing a fleche `Cache` to a `RunConfig` lets decorated nodes hit the same cache
  during a workflow run.
- **Signed storage**: HMAC-sign pickle-family entries with a `secret_key`; tampered or
  wrong-key entries surface as plain cache misses, never as executed code.


- Caching input and output files or directories does not work yet, but is actively developed.

## Where to go next

- Docs: <https://fleche.readthedocs.io>
- Source: <https://github.com/pmrv/fleche>
- Deeper notebooks in this directory: `GettingStarted`, `ExtraMethods`,
  `StorageBackends`, `ConcurrentExecution`, `CacheStack`, `SecureStorage`,
  `TransferWorkflow`.